In [1]:
#Global variables for reproducibility
random_state = 269
kaggle_username = 'sarahc0154'
kaggle_API_key = 'KGAT_f93980b6e5a9e36282a7d8a0773fb7ef'

#Anshul's info
#kaggle_username = 'brucewaynedied12'
#kaggle_API_key = 'KGAT_984e7e5a17a41c75481e1be9d2c84b20'


In [2]:
!pip install opencv-python
!pip install keras-tuner

KeyboardInterrupt: 

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import cv2
import shutil
import seaborn as sns
import tensorflow as tf
import keras_tuner as kt

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from google.colab.patches import cv2_imshow
from sklearn.preprocessing import LabelBinarizer, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import train_test_split

In [ ]:
#Upload data
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_API_key
!kaggle datasets download -d sumithsingh/blood-cell-images-for-cancer-detection

#Unzip file
!unzip -q blood-cell-images-for-cancer-detection.zip -d blood_cell_data

In [ ]:
#Get HSV values associated with purple (color of nucleus)
purple = np.uint8([[[128,0,128 ]]])
hsv_purple = cv2.cvtColor(purple,cv2.COLOR_BGR2HSV)
print(hsv_purple)

In [ ]:
#Image preprocessing for logistic regression
original_dir = 'blood_cell_data'
preprocessed_dir_logreg = 'preprocessed_data_logreg'

if os.path.exists(preprocessed_dir_logreg):
  shutil.rmtree(preprocessed_dir_logreg)
os.makedirs(preprocessed_dir_logreg)

for folder in os.listdir(original_dir):
    orig_folder_path = os.path.join(original_dir, folder)
    proc_folder_path = os.path.join(preprocessed_dir_logreg, folder)

    for img_file in os.listdir(orig_folder_path):
      img_path = os.path.join(orig_folder_path, img_file)
      img = cv2.imread(img_path)

      img_blur_logreg = cv2.GaussianBlur(img, (5, 5), 0)

      hsv_blur_logreg = cv2.cvtColor(img_blur_logreg, cv2.COLOR_BGR2HSV)
      mask_lower_limit = np.array([130,50,108])
      mask_upper_limit = np.array([170,255,148])
      mask_for_logreg = cv2.inRange(hsv_blur_logreg, mask_lower_limit, mask_upper_limit)

      canny = cv2.Canny(mask_for_logreg, 100, 200)

      img_resized_logreg = cv2.resize(canny, (224, 224))

      save_path = os.path.join(proc_folder_path, img_file)
      os.makedirs(proc_folder_path, exist_ok=True)
      cv2.imwrite(save_path, img_resized_logreg)

In [ ]:
#Image preprocessing for CNN
original_dir = 'blood_cell_data'
preprocessed_dir_cnn = 'preprocessed_data_cnn'

if os.path.exists(preprocessed_dir_cnn):
  shutil.rmtree(preprocessed_dir_cnn)
os.makedirs(preprocessed_dir_cnn)

for folder in os.listdir(original_dir):
    orig_folder_path = os.path.join(original_dir, folder)
    proc_folder_path = os.path.join(preprocessed_dir_cnn, folder)

    for img_file in os.listdir(orig_folder_path):
      img_path = os.path.join(orig_folder_path, img_file)
      img = cv2.imread(img_path)

      hsv_cnn = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
      mask_lower_limit = np.array([130,50,108])
      mask_upper_limit = np.array([170,255,148])
      mask_for_cnn = cv2.inRange(hsv_cnn, mask_lower_limit, mask_upper_limit)

      img_resized_cnn = cv2.resize(mask_for_cnn, (224, 224))

      save_path = os.path.join(proc_folder_path, img_file)
      os.makedirs(proc_folder_path, exist_ok=True)
      cv2.imwrite(save_path, img_resized_cnn)

In [ ]:
#Visualizing differences in preprocessing
original_img_path = os.path.join(original_dir, 'basophil', 'BA_100102.jpg')
orignal_img = cv2.imread(original_img_path)
original_img_proper_colors = cv2.cvtColor(orignal_img, cv2.COLOR_BGR2RGB)
img_logreg = os.path.join(preprocessed_dir_logreg, 'basophil', 'BA_100102.jpg')
img_cnn = os.path.join(preprocessed_dir_cnn, 'basophil', 'BA_100102.jpg')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Image Preprocessing Comparison\n', fontsize=16)

axes[0].imshow(original_img_proper_colors)
axes[0].set_title('Original Image')
axes[0].axis('off')

# Preprocessed for Logistic Regression
axes[1].imshow(cv2.imread(img_logreg))
axes[1].set_title('Logistic Regression Preprocessing')
axes[1].axis('off')

# Preprocessed for CNN
axes[2].imshow(cv2.imread(img_cnn))
axes[2].set_title('CNN Preprocessing')
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
#Set up for multinomial regression and hyperparameter tuning
X = []
y = []

for folder in os.listdir(preprocessed_dir_logreg):
  if folder != 'erythroblast':
    folder_path = os.path.join(preprocessed_dir_logreg, folder)
      #Excludes red blood cells

    for img_file in os.listdir(folder_path):
      img_path = os.path.join(folder_path, img_file)
      img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
      if img is not None:
        X.append(img.flatten())
        y.append(folder)
        del img
          #Saves RAM

X = np.array(X, dtype = 'uint8')
y = np.array(y)

#Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = random_state, stratify = y)

#Normalizes pixel values between 0 and 1
X_train = X_train / 255
X_test = X_test / 255

In [ ]:
#Hyperparameter tuning (may take a while to run; not necessary to run this cell)
C_values = np.logspace(-4, 4, 10)
for C in C_values:
    tuning_model = LogisticRegression(penalty = 'l2', C = C)
    tuning_model.fit(X_train, y_train)
    training_acc = tuning_model.score(X_train, y_train)
    test_acc = tuning_model.score(X_test, y_test)
    print(f'{C:<30}{training_acc:<30}{test_acc:<30}')
#Test accuracy peaks at C = 0.005994842503189409

In [ ]:
#Applying the multinomial model
unique_elements, counts = np.unique(y, return_counts=True)

logreg = LogisticRegression(multi_class = "multinomial", penalty = 'l2', solver = "lbfgs", max_iter = 1000, C = 0.005994842503189409)
logreg.fit(X_train, y_train)

logreg_prediction = logreg.predict(X_test)

logreg_prob = logreg.predict_proba(X)
logreg_roc_score = roc_auc_score(y, logreg_prob, multi_class='ovr', average='macro')
  #Use macro because classes are equally represented

#Outputting results
print("Class Distribution")
for element, count in zip(unique_elements, counts):
    print(f"{element}: {count}")

print("\nAccuracy:")
print(accuracy_score(y_test, logreg_prediction))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, logreg_prediction))

print("\nClassification Report:")
print(classification_report(y_test, logreg_prediction))

print("\nAUC Score:")
print(logreg_roc_score)

In [ ]:
#Visualization of multinomial regression results
#Confusion matrix
conf_matrix = confusion_matrix(y_test, logreg_prediction)
conf_matrix_df = pd.DataFrame(conf_matrix, index = logreg.classes_, columns = logreg.classes_)

plt.figure(figsize = (8, 6))
sns.heatmap(conf_matrix_df, annot = True, fmt = 'd', cmap = 'PuRd', cbar = True)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix for Multinomial Regression')
plt.show()
  #Model most likely to confuse basophils and monocytes, as well as neutrophils and myeloblasts. Model is best at identifying myeloblasts (which is focus of project).

#ROC curve
label_binarizer = LabelBinarizer().fit(y_train)
y_onehot_test = label_binarizer.transform(y_test)
y_onehot_test.shape
  #y_test is converted to have the same shape (800, 4) as logreg_prob_test

logreg_prob_test = logreg.predict_proba(X_test)

class_id = []
for i in range(len(label_binarizer.classes_)):
  class_id.append(np.flatnonzero(label_binarizer.classes_ == label_binarizer.classes_[i])[0])

fig, ax = plt.subplots(figsize=(6, 6))

colors = ['powderblue', 'mediumpurple', 'palevioletred', 'crimson']
target_names = ['basophil', 'monocyte', 'myeloblast', 'seg_neutrophil']
for class_id, color in zip(range(len(class_id)), colors):
    RocCurveDisplay.from_predictions(
        y_onehot_test[:, class_id],
        logreg_prob_test[:, class_id],
        name = f"ROC curve for {target_names[class_id]}",
        color = color,
        ax = ax,
        plot_chance_level=(class_id == 3),
        despine=True,
    )

ax.set(
    xlabel = "False Positive Rate",
    ylabel = "True Positive Rate",
    title = "One-versus-Rest ROC Curve for Multinomial Regression",
)

In [ ]:
#Setup for CNN
X_cnn = []
y_cnn = []

for folder in os.listdir(preprocessed_dir_cnn):
    if folder != 'erythroblast':
        folder_path = os.path.join(preprocessed_dir_cnn, folder)
        #Excludes red blood cells

        for img_file in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                X_cnn.append(img)
                y_cnn.append(folder)
                del img  #Saves RAM

X_cnn = np.array(X_cnn, dtype='uint8')
y_cnn = np.array(y_cnn)

X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(X_cnn, y_cnn, test_size = 0.2, random_state = random_state, stratify = y_cnn)

X_validation_cnn, X_test_cnn, y_validation_cnn, y_test_cnn = train_test_split(X_test_cnn, y_test_cnn, test_size = 0.5, random_state = random_state, stratify = y_test_cnn)
#Overall: 80% train, 10% validation, 10% test

#Normalize pixel values between 0 and 1
X_train_cnn = X_train_cnn / 255
X_validation_cnn = X_validation_cnn / 255
X_test_cnn = X_test_cnn / 255

#Reshape for CNN input
X_train_cnn = X_train_cnn.reshape(-1, 224, 224, 1)
X_validation_cnn = X_validation_cnn.reshape(-1, 224, 224, 1)
X_test_cnn = X_test_cnn.reshape(-1, 224, 224, 1)

#One-hot encode labels
label_binarizer_cnn = LabelBinarizer()
y_train_cnn = label_binarizer_cnn.fit_transform(y_train_cnn)
y_validation_cnn = label_binarizer_cnn.transform(y_validation_cnn)
y_test_cnn = label_binarizer_cnn.transform(y_test_cnn)

In [ ]:
#Define the model and prepare it for hyperparameter tuning
def model_builder(hp):
    cnn_model = Sequential()

    activation_function_choice = hp.Choice('activation_function', values=['relu', 'tanh'])

    cnn_model.add(Conv2D(
        filters = hp.Int('layer_one_neurons', min_value = 16, max_value = 128, step = 16),
        kernel_size = (3, 3),
        activation = activation_function_choice,
        input_shape = (224, 224, 1)
        #Input shape is (224, 224, 1) instead of (224, 224, 3) because we are using grayscale images
    ))
    cnn_model.add(MaxPooling2D())

    cnn_model.add(Conv2D(
        filters = hp.Int('layer_two_neurons', min_value = 32, max_value = 128, step=16),
        kernel_size = (3, 3),
        activation = activation_function_choice
    ))
    cnn_model.add(MaxPooling2D())

    cnn_model.add(Conv2D(
        filters = hp.Int('layer_three_neurons', min_value = 32, max_value = 128, step = 16),
        kernel_size = (3, 3),
        activation = activation_function_choice
    ))
    cnn_model.add(MaxPooling2D())

    cnn_model.add(Flatten())

    cnn_model.add(Dense(
        units = hp.Int('dense_layer_neurons', min_value = 64, max_value = 256, step = 16),
        activation = activation_function_choice
    ))

    cnn_model.add(Dropout(
        rate = hp.Float('dropout_rate', min_value = 0.2, max_value = 0.5, step = 0.1)
    ))

    cnn_model.add(Dense(4, activation = 'softmax'))
    #Four total classes
    #Softmax used to convert logit values to probability distribution

    cnn_model.compile(
        optimizer = 'adam',
        loss = 'categorical_crossentropy',
        metrics = ['accuracy']
    )

    return cnn_model

In [ ]:
#Hyperparameter tuning
tuner = kt.Hyperband(
    model_builder,
    objective = 'val_accuracy',
    max_epochs = 10,
    factor = 3,
    seed = random_state,
    directory = 'cnn_tuning',
    project_name = 'blood_cell_cnn'
)

tuner.search(
    X_train_cnn, y_train_cnn,
    epochs=10,
    validation_data=(X_validation_cnn, y_validation_cnn)
)

#Applying the CNN model
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
model = tuner.hypermodel.build(best_hp)
model.fit(X_train_cnn, y_train_cnn, epochs=10, validation_data=(X_validation_cnn, y_validation_cnn))

In [ ]:
#Outputting CNN Results
test_loss, test_acc = model.evaluate(X_test_cnn, y_test_cnn, verbose=0)
print("Accuracy:")
print(f"{test_acc:.2f}")

test_prediction_prob = model.predict(X_test_cnn)
test_prediction = np.argmax(test_prediction_prob, axis=1)
true_test_labels = np.argmax(y_test_cnn, axis=1)

#Confusion matrix
conf_matrix_cnn = confusion_matrix(true_test_labels, test_prediction)
conf_matrix_cnn_df = pd.DataFrame(conf_matrix_cnn, index=label_binarizer_cnn.classes_, columns=label_binarizer_cnn.classes_)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_cnn_df, annot=True, fmt='d', cmap='PuRd', cbar=True)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix for CNN')
plt.show()

#Classification report
print("\nClassification Report:")
print(classification_report(true_test_labels, test_prediction, target_names=label_binarizer_cnn.classes_))

#AUC score
y_onehot_cnn = y_test_cnn
cnn_roc_score = roc_auc_score(y_onehot_cnn, test_prediction_prob, multi_class='ovr', average='macro')
print("\nAUC Score:")
print(cnn_roc_score)

#ROC curve
fig, ax = plt.subplots(figsize=(6, 6))
for i, (color, name) in enumerate(zip(['powderblue', 'mediumpurple', 'palevioletred', 'crimson'], label_binarizer_cnn.classes_)):
    RocCurveDisplay.from_predictions(y_onehot_cnn[:, i], test_prediction_prob[:, i], name=name, color=color, ax=ax, plot_chance_level=(i==3))
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="One-versus-Rest ROC Curve for CNN")
plt.show()

In [ ]:
import pickle

model.save('blood_cell_cnn_model.keras')

with open('label_binarizer_cnn.pkl', 'wb') as f:
    pickle.dump(label_binarizer_cnn, f)

In [ ]:
!pip install -q streamlit


In [ ]:
%%writefile streamlit_app.py
import streamlit as st
!pip install tensorflow
import tensorflow as tf
import numpy as np
import cv2
from PIL import Image
import pickle

@st.cache_resource
def load_my_model():
    model = tf.keras.models.load_model('blood_cell_cnn_model.keras')
    return model

model = load_my_model()

with open('label_binarizer_cnn.pkl', 'rb') as f:
    label_binarizer_cnn = pickle.load(f)

def preprocess_image(image):
    img_array = np.array(image.convert('RGB'))
    hsv_cnn = cv2.cvtColor(img_array, cv2.COLOR_RGB2HSV)
    mask_lower_limit = np.array([130, 50, 108])
    mask_upper_limit = np.array([170, 255, 148])
    mask_for_cnn = cv2.inRange(hsv_cnn, mask_lower_limit, mask_upper_limit)
    img_resized_cnn = cv2.resize(mask_for_cnn, (224, 224))
    img_normalized = img_resized_cnn / 255.0
    return img_normalized.reshape(1, 224, 224, 1)

st.title('Blood Cell Type Classification')
st.write('Upload a blood cell image to classify its type.')

uploaded_file = st.file_uploader("Choose an image...", type=['jpg', 'jpeg', 'png'])

if uploaded_file is not None:
    image = Image.open(uploaded_file)
    st.image(image, caption='Uploaded Image', use_column_width=True)
    st.write("")
    st.write("Classifying...")

    processed_image = preprocess_image(image)
    prediction = model.predict(processed_image)
    predicted_class_idx = np.argmax(prediction)
    predicted_class = label_binarizer_cnn.classes_[predicted_class_idx]
    confidence = prediction[0][predicted_class_idx] * 100

    st.success(f'Prediction: {predicted_class} with {confidence:.2f}% confidence.')

    st.write("### All Class Probabilities:")
    probabilities_df = pd.DataFrame({
        'Class': label_binarizer_cnn.classes_,
        'Probability': prediction[0]
    }).sort_values(by='Probability', ascending=False)
    st.dataframe(probabilities_df)